## Jupyter Run Time

When you execute a cell, the kernel process stays running. When you execute the next cell, it runs in the **same process** — same Python interpreter, same namespace.

That's why:

- Variables from Cell 1 are available in Cell 2
- Imports persist across cells
- State accumulates (and why "Restart Kernel" clears everything)

Each cell is just executing code in the same kernel's `__main__` module. The process never stops between cells — it waits for your next input.

---

### Module initialization and Run Time
**Every module gets exactly one init per process**, registered in `sys.modules` under its qualified name, and every subsequent import — no matter how many places, how many times — is a dict lookup that returns the same object. That holds for stdlib, third-party, and your own modules alike. The 5-step lifecycle runs once per unique `sys.modules` key, never per `import` statement.

**Why does pip/poetry-installed code "just work" with `import`?**

There's no separate import mechanism for third-party packages — that's the key insight. `site-packages` is importable for the **same reason** your own project folder is: it's a directory on `sys.path`, and `PathFinder` walks `sys.path` looking for a match. The only special thing is _how it gets onto_ `sys.path`:

In [1]:
import sys

print([p for p in sys.path if 'site-packages' in p])

['c:\\Users\\Dennie\\Mate Academy\\Python Core\\python_essentials\\.venv\\Lib\\site-packages']


At interpreter startup, Python auto-imports a module called `site` (unless you run `python -S`). `site.py` finds your environment's `site-packages` directory and appends it to `sys.path`. Once it's there, `pkg_a` and `requests` are found by the exact same `PathFinder` mechanism, in the exact same loop. `pip install` and `poetry install` do nothing magical — they just copy files into a directory that `site` has already wired into the search path.

**The `__main__` subtlety**
Entry-point module is **added to `sys.modules` cache** under the key `'__main__'`, _not_ under the module's real filename. Step 4 (register-before-execute) still applies — `sys.modules['__main__']` is set before the file body runs, same as any import.

The trap: if some other code later does `import app` (where `app.py` is the file you ran directly), Python doesn't recognize it as "the same module already running." It has no entry for `'app'` in `sys.modules`, so it finds `app.py` on disk, creates a **second, distinct module object**, and executes the entire file **again** — top to bottom, **all module-level code re-runs**.

In [3]:
# app.py
print("app.py module-level code running")

if __name__ == "__main__":
    print("running as entry point")

app.py module-level code running
running as entry point


Run `python app.py` → both lines print, `sys.modules['__main__']` exists. If a sibling module does `import app` → `app.py`'s top-level `print` fires _again_ (now under `sys.modules['app']`), but the `if __name__ ==` block stays silent — `__name__` is `'app'` there, not `'__main__'`. Two separate module objects, two separate executions, in the same process.

This is exactly why the guard exists: to keep entry-point-only behavior (CLI parsing, `app.run()`) from firing during that second, accidental execution.

> **One-liner to file away:** _"Modules are cached by name, not by file — the entry-point script lives in `sys.modules` as `'__main__'`, so re-importing it by its real name re-executes the whole file as a second, independent module object."_

---

## The Python Module System — Runtime Model

**Restart kernel → Run All** for the most instructive output.  
Print-tracing in the demo packages shows exactly when Python executes each file.

In [1]:
import sys
from pathlib import Path

_demo = Path.cwd() / 'demo_packages'
if not _demo.exists():
    _demo = Path.cwd() / 'modules_and_imports' / 'demo_packages'
assert _demo.exists(), f'demo_packages not found — expected: {_demo}'

if str(_demo) not in sys.path:
    sys.path.insert(0, str(_demo))
print(f'demo_packages on path: {sys.path[0]}')

demo_packages on path: c:\Users\Dennie\Mate Academy\Python Core\python_essentials\modules_and_imports\demo_packages


## 1. A module is a **runtime object**

Before any import there is no "module" — just a `.py` file on disk.  
When Python imports it, it:
- creates a `types.ModuleType` object
- populates its `__dict__`
- caches it in `sys.modules`.

| Attribute | Meaning |
|---|---|
| `__name__` | Fully qualified name: `'pkg_a.core'` |
| `__file__` | Absolute path to the `.py` source file |
| `__package__` | Parent package name, or `''` for top-level modules |
| `__spec__` | `ModuleSpec` — blueprint used to create the module |
| `__path__` | **Packages only** — dirs Python searches for submodules |
| `__loader__` | Object that read and executed the source |
| `__dict__` | The module's namespace — every `def`/`class`/assignment lives here |

In [2]:
import os, types

print(f'type(os)                         = {type(os)}')
print(f'isinstance(os, types.ModuleType) = {isinstance(os, types.ModuleType)}')
print()
for attr in ['__name__', '__file__', '__package__', '__spec__', '__loader__', '__path__']:
    val = getattr(os, attr, 'N/A')
    print(f'os.{attr:<15} = {val!r}')

type(os)                         = <class 'module'>
isinstance(os, types.ModuleType) = True

os.__name__        = 'os'
os.__file__        = 'C:\\Python313\\Lib\\os.py'
os.__package__     = ''
os.__spec__        = ModuleSpec(name='os', loader=<class '_frozen_importlib.FrozenImporter'>, origin='frozen')
os.__loader__      = <class '_frozen_importlib.FrozenImporter'>
os.__path__        = 'N/A'


In [1]:
# json is a real package (json/ directory with __init__.py); os is a plain module (os.py)
import json
import os

has_pkg = hasattr(json, "__path__")
has_mod = hasattr(os, "__path__")

print(f"json  (package) — __path__ present: {has_pkg}")
print(f"  __path__ = {json.__path__}")
print(f"  __package__ = {json.__package__}")
print()
print(f"os    (module)  — __path__ present: {has_mod}")
print(f"  __name__    = {os.__name__!r}")
print(f"  __package__ = {os.__package__!r}")

json  (package) — __path__ present: True
  __path__ = ['C:\\Python313\\Lib\\json']
  __package__ = json

os    (module)  — __path__ present: False
  __name__    = 'os'
  __package__ = ''


## 2. The 5-step import lifecycle

When Python executes `import pkg_a.core`:

```shell
Step 1 — CHECK sys.modules
         'pkg_a.core' already there?  YES → return cached object.  DONE.
                                       NO  → continue.

Step 2 — FIND
         Walk sys.meta_path finders: [BuiltinImporter, FrozenImporter, PathFinder]
         PathFinder searches sys.path dirs for pkg_a/core.py

Step 3 — LOAD
         Create types.ModuleType('pkg_a.core')
         Set __name__, __file__, __package__, __spec__, __loader__

Step 4 — REGISTER  (before executing the file)
         sys.modules['pkg_a.core'] = <new, still-empty module>
         ← The stub is in sys.modules BEFORE the file body runs.
           This is why circular imports partially work.

Step 5 — EXECUTE
         Run pkg_a/core.py in the module's __dict__.
         Every def / class / assignment adds a name to __dict__.
```

Steps 1–5 run once. Every subsequent `import` only runs Step 1.

In [1]:
# Fresh kernel: the [once] init print appears — Steps 1-5 run.
# Run again without restarting: silence — Step 1 returns the cache.

import demo_packages.lifecycle_demo.once as once_mod

print()
print(f'__name__    = {once_mod.__name__!r}')
print(f'__file__    = {once_mod.__file__!r}')
print(f'__package__ = {once_mod.__package__!r}')

[lifecycle_demo.__init__] initializing
[lifecycle_demo.once] ← this prints ONCE, no matter how many importers exist

__name__    = 'demo_packages.lifecycle_demo.once'
__file__    = 'c:\\Users\\Dennie\\Mate Academy\\Python Core\\python_essentials\\modules_and_imports\\demo_packages\\lifecycle_demo\\once.py'
__package__ = 'demo_packages.lifecycle_demo'


In [2]:
# Run this again — no init print = sys.modules cache was hit
import demo_packages.lifecycle_demo.once
print('Done — silence above means the cache was used, no re-execution')

Done — silence above means the cache was used, no re-execution


## 3. `sys.path` — where Python looks

`sys.path` is a list of directory strings searched left-to-right. First match wins.

| Typical entry | Source |
|---|---|
| `sys.path[0]` | Dir of the running script (`''` = cwd in interactive/notebook mode) |
| `PYTHONPATH` entries | Injected from the OS env var at process startup |
| stdlib dirs | Python's own standard library |
| `site-packages` | Third-party packages installed by `pip` |

In [6]:
import sys

print(f'sys.path ({len(sys.path)} entries):')
for i, entry in enumerate(sys.path[:8]):
    print(f'  [{i}] {entry!r}')
if len(sys.path) > 8:
    print(f'  ... and {len(sys.path) - 8} more')

sys.path (8 entries):
  [0] 'c:\\Users\\Dennie\\Mate Academy\\Python Core\\python_essentials\\modules_and_imports\\demo_packages'
  [1] 'C:\\Python313\\python313.zip'
  [2] 'C:\\Python313\\DLLs'
  [3] 'C:\\Python313\\Lib'
  [4] 'C:\\Python313'
  [5] 'c:\\Users\\Dennie\\Mate Academy\\Python Core\\python_essentials\\.venv'
  [6] ''
  [7] 'c:\\Users\\Dennie\\Mate Academy\\Python Core\\python_essentials\\.venv\\Lib\\site-packages'


In [7]:
# The three default finders Python installs at startup
for finder in sys.meta_path:
    print(f'{type(finder).__name__:<30} from {type(finder).__module__}')

# BuiltinImporter  — C extensions compiled into the interpreter
# FrozenImporter   — bytecode frozen at compile time (zipimport)
# PathFinder       — searches sys.path for .py files and packages (most common)

type                           from builtins
type                           from builtins
type                           from builtins
_SixMetaPathImporter           from six


## 4. sys.modules — the import cache

`sys.modules` is a plain `dict[str, types.ModuleType]`.

**Key behaviors:**
- Importing the same module twice returns the cached object — no re-execution
- All importers share the **same object** — mutation in one place is visible everywhere
- `del sys.modules['X']` forces re-import on the next `import X` (almost never intentional)
- `importlib.reload(module)` re-executes the file and updates `__dict__` in place

In [8]:
import sys

# Browse the cache — filter to our demo modules
prefixes = ('lifecycle_demo', 'pkg_a', 'pkg_b')
interesting = sorted(
    k for k in sys.modules
    if k.startswith(prefixes) and sys.modules[k] is not None
)
for name in interesting:
    mod = sys.modules[name]
    print(f'{name:<40}  id={id(mod)}')

In [9]:
# Two import statements → one shared object
import lifecycle_demo.once
import lifecycle_demo.once as once2

print(f'Same object: {lifecycle_demo.once is once2}')
cached = 'lifecycle_demo.once' in sys.modules
print(f'In sys.modules: {cached}')
print()
# _call_count is shared state — the same module object is used by ALL importers
print(f'call 1: {once2.get_count()}')
print(f'call 2: {once2.get_count()}')
print(f'call 3: {once2.get_count()}')

[lifecycle_demo.__init__] initializing
[lifecycle_demo.once] ← this prints ONCE, no matter how many importers exist
Same object: True
In sys.modules: True

call 1: 1
call 2: 2
call 3: 3


In [10]:
# importer1 triggers once.py initialization (first load).
# importer2 ALSO imports once.py — but it's already cached.
# The init print appears ONCE across both importers.

import lifecycle_demo.importer1
print('--- importing importer2 ---')
import lifecycle_demo.importer2

[importer1] loading, importing once...
[importer1] done. get_count() = 4
--- importing importer2 ---
[importer2] loading, importing once...
[importer2] done. get_count() = 5


## 5. PYTHONPATH

`PYTHONPATH` is an OS environment variable (`;`-separated on Windows, `:`-separated on Unix).  
At process startup, each directory is prepended to `sys.path` — before stdlib, after `sys.path[0]`.

```bash
# Make a local package importable without pip install -e .
export PYTHONPATH=/home/user/myproject   # Unix
set PYTHONPATH=C:\Users\user\myproject   # Windows
```

**Rules:**
- Development: add the project root so top-level packages are importable
- Inside a virtualenv: let the venv manage `sys.path`; don't override with `PYTHONPATH`
- Django/FastAPI: `manage.py` and `uvicorn` configure `sys.path` for you — no manual setting needed

In [11]:
import os

pythonpath = os.environ.get('PYTHONPATH', '(not set)')
print(f'PYTHONPATH = {pythonpath!r}')
print()
print('First 5 entries of sys.path:')
for entry in sys.path[:5]:
    print(f'  {entry!r}')

PYTHONPATH = '(not set)'

First 5 entries of sys.path:
  'c:\\Users\\Dennie\\Mate Academy\\Python Core\\python_essentials\\modules_and_imports\\demo_packages'
  'C:\\Python313\\python313.zip'
  'C:\\Python313\\DLLs'
  'C:\\Python313\\Lib'
  'C:\\Python313'


## 6. importlib — programmatic import machinery

| Tool | Use case |
|---|---|
| `importlib.util.find_spec(name)` | Locate a module without importing it |
| `importlib.import_module(name)` | Import by string name — plugin systems, config-driven loading |
| `importlib.reload(module)` | Re-execute the source file; update `__dict__` in place |

In [12]:
import importlib.util

spec = importlib.util.find_spec('lifecycle_demo.once')
print(f'name:    {spec.name!r}')
print(f'origin:  {spec.origin!r}')   # absolute path to the .py file
print(f'loader:  {type(spec.loader).__name__}')
print(f'submodule_search_locations: {spec.submodule_search_locations}')

name:    'lifecycle_demo.once'
origin:  'c:\\Users\\Dennie\\Mate Academy\\Python Core\\python_essentials\\modules_and_imports\\demo_packages\\lifecycle_demo\\once.py'
loader:  SourceFileLoader
submodule_search_locations: None


In [13]:
import importlib

# import_module — import by string name (useful when name is determined at runtime)
mod_name = 'lifecycle_demo.once'
m = importlib.import_module(mod_name)
print(f'Imported: {m.__name__!r}')
print(f'get_count() = {m.get_count()}')

Imported: 'lifecycle_demo.once'
get_count() = 6


In [14]:
import importlib
import lifecycle_demo.once

# reload() re-executes the source file.
# The init print fires again. __dict__ is updated in-place.
# _call_count resets to 0 because module-level code re-ran.
# See 04_interview_traps for the reload gotcha (stale references).
print('--- before reload ---')
importlib.reload(lifecycle_demo.once)
print('--- after reload ---')

--- before reload ---
[lifecycle_demo.once] ← this prints ONCE, no matter how many importers exist
--- after reload ---
